# NER 微调训练（bert-base-chinese，Colab T4）

对应算法设计 §2.3，训练数据 HF `ttxy/resume_ner`（3821/463/477 句）。
全部自动：克隆仓库 → 装依赖 → 下载数据集 → 训练 → 打包下载。

**用法（3 步）**
1. 菜单栏：代码执行程序 → 更改代码执行程序类型 → **T4 GPU** → 保存
2. 菜单栏：代码执行程序 → **全部运行**
3. 等 20~40 分钟，最后自动弹出 `ner_model.zip` 下载

拿到 zip 后：解压到项目的 `backend/models/ner/`（config.json 直接位于 ner/ 下），重启后端即自动启用 BERT 实体抽取，无需改代码。
之后在 backend 目录重跑 `python tools/run_experiments.py` 可复测三组对比（DeepSeek 组命中缓存，零成本）。

In [ ]:
# 0) 确认 GPU（没开 T4 就先改运行时类型再重跑本格）
import torch
if torch.cuda.is_available():
    print("GPU 就绪:", torch.cuda.get_device_name(0))
else:
    print("⚠ 未检测到 GPU——请到「代码执行程序→更改代码执行程序类型→T4 GPU」后再全部运行")

In [ ]:
# 1) 克隆项目仓库（已存在则跳过，可重复执行）
import os
if not os.path.exists("/content/resume-matcher"):
    !git clone --depth 1 https://github.com/CZT123526/resume-matcher.git /content/resume-matcher
!ls /content/resume-matcher/backend/tools

In [ ]:
# 2) 装依赖（torch 用 Colab 自带的 CUDA 版，不必重装）
!pip -q install -U "transformers>=4.41" "datasets>=2.19" "accelerate>=1.1.0" httpx

In [ ]:
# 3) 下载并转换 NER 数据集（hf-mirror 拉取，脚本自校验 3821/463/477）
!python /content/resume-matcher/backend/tools/data/download_dataset.py

In [ ]:
# 4) 训练（4 epoch，T4 约 20~40 分钟；产物 = 模型 + tokenizer + metrics.json）
import os
os.environ["NER_DATA_DIR"] = "/content/resume-matcher/backend/tools/data"
os.environ["NER_MODEL_DIR"] = "/content/models_ner"
!cd /content/resume-matcher/backend && python tools/train_ner.py

In [ ]:
# 5) 剔除训练中间检查点，打包并触发浏览器下载
!cd /content && rm -rf models_ner/_ckpt && zip -qr ner_model.zip models_ner && ls -lh ner_model.zip
from google.colab import files
files.download("ner_model.zip")

## 收尾（回到本机）
1. 解压 `ner_model.zip` → 把 `models_ner/` 里的内容放到 `backend/models/ner/`（config.json 直接在 ner/ 下）
2. 重启后端 → `/api/model/info` 的实体抽取来源变为 BERT 微调模型
3. `backend` 目录下 `python tools/run_experiments.py` 复测三组对比，预期 NDCG 提升

遇到断连：Colab 免费版长时间空闲会回收实例，重开本页全部重跑即可（流程幂等）。